# Experiment 2.10: Anti-sharpness toy test — endpoint finite-difference Hessian comparison

**Counterpart script:** `experiments/Tier_1_Core_Mechanism_Tests/ANTI_SHARPNESS_hessian_trace/run_experiment.py`

This notebook is the analysis/reporting front-end for the paired script above. It **imports and uses the script's returned results** rather than re-implementing the experiment core.

## Question

In a small deep-linear toy problem, does **Muon** converge to a **lower endpoint Hessian trace** than **SGD with momentum**?

## What this notebook measures

- final losses and steps-to-threshold
- a **full finite-difference Hessian estimate** at the converged endpoint
- endpoint spectral summaries: trace, $\lambda_{\max}$, effective rank, near-zero eigenvalue count **proxy**, condition number, and negative-curvature diagnostics

## What this notebook does **not** measure

- training-trajectory sharpness or movement through sharp directions
- a true gauge-subspace dimension
- generalization

The near-zero eigenvalue count should be treated as a **threshold-based proxy**, not as a direct gauge measurement.


In [ ]:
import importlib.util
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from IPython.display import HTML, Markdown, display
except ImportError:
    class _PlainDisplayString(str):
        pass

    def Markdown(text):
        return _PlainDisplayString(text)

    def HTML(text):
        return _PlainDisplayString(text)

    def display(obj):
        print(obj)


In [ ]:
def find_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    relative_script = Path('experiments/Tier_1_Core_Mechanism_Tests/ANTI_SHARPNESS_hessian_trace/run_experiment.py')
    for root in candidates:
        if (root / relative_script).exists():
            return root
    raise FileNotFoundError(
        'Could not locate repo root containing '
        'experiments/Tier_1_Core_Mechanism_Tests/ANTI_SHARPNESS_hessian_trace/run_experiment.py'
    )


REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'ANTI_SHARPNESS_hessian_trace'
SCRIPT_PATH = EXPERIMENT_DIR / 'run_experiment.py'
NOTEBOOK_PATH = EXPERIMENT_DIR / 'run_experiment.ipynb'

spec = importlib.util.spec_from_file_location('anti_sharpness_hessian_trace', SCRIPT_PATH)
anti = importlib.util.module_from_spec(spec)
spec.loader.exec_module(anti)

display(Markdown(
    f"""
## Pair identity and notebook-safe import setup

- **Repository root:** `{REPO_ROOT}`
- **Notebook path:** `{NOTEBOOK_PATH.relative_to(REPO_ROOT)}`
- **Script path:** `{SCRIPT_PATH.relative_to(REPO_ROOT)}`
- **Import mode:** `importlib.util.spec_from_file_location(...)`

This avoids notebook dependence on `__file__` and keeps the script as the computation source of truth.
"""
))


## Reproducibility, runtime, and configuration

The cell below runs the paired script through its import-safe `run_experiment(...)` function and logs the environment, seeds, configuration, and observed runtime.


In [ ]:
wall_start = time.time()
results = anti.run_experiment(verbose=False)
wall_elapsed = time.time() - wall_start

ARCH_KEYS = list(results.get('architecture_order', results['architectures'].keys()))
CONFIG = results['config']
SEEDS = results['seeds']

runtime_lines = [
    '## Run metadata',
    '',
    f"- **Study ID:** {results['study_id']}",
    f"- **Title:** {results['title']}",
    f"- **Question:** {results['question']}",
    f"- **Python:** `{sys.version.split()[0]}`",
    f"- **Platform:** `{platform.platform()}`",
    f"- **NumPy:** `{np.__version__}`",
    f"- **Notebook wall-clock runtime:** `{wall_elapsed:.2f} s`",
    f"- **Returned experiment runtime:** `{results['overall']['runtime_seconds']:.2f} s`",
    f"- **Seeds:** `{SEEDS}`",
    '',
    '### Default toy configuration',
    '',
    f"- `dim = {CONFIG['dim']}`",
    f"- `num_steps = {CONFIG['num_steps']}`",
    f"- `lr_sgd = {CONFIG['lr_sgd']}`",
    f"- `lr_muon = {CONFIG['lr_muon']}`",
    f"- `momentum = {CONFIG['momentum']}`",
    f"- `ns_iters = {CONFIG['ns_iters']}`",
    f"- `convergence_factor = {CONFIG['convergence_factor']}`",
    f"- `hessian_eps = {CONFIG['hessian_eps']}`",
    f"- `eff_rank_threshold = {CONFIG['eff_rank_threshold']} * lambda_max`",
    f"- `near_zero_threshold = {CONFIG['near_zero_threshold']} * lambda_max`",
]
display(Markdown('\n'.join(runtime_lines)))


In [ ]:
def metric_summary(arch_key, metric):
    return results['architectures'][arch_key]['metric_summaries'][metric]


def seed_results(arch_key):
    return results['architectures'][arch_key]['seed_results']


def per_seed_metric(arch_key, optimizer, metric):
    return np.asarray([seed_result[optimizer][metric] for seed_result in seed_results(arch_key)], dtype=float)


def format_value(value, kind='scientific'):
    if value is None:
        return 'None'
    if isinstance(value, (bool, np.bool_)):
        return str(bool(value))
    if not np.isfinite(value):
        return '∞' if value == np.inf else 'NaN'
    if kind == 'count':
        return f'{value:.1f}'
    if kind == 'float':
        return f'{value:.2f}'
    return f'{value:.4e}'


def mean_std_text(summary, metric):
    kind = anti.METRIC_DISPLAY[metric]['kind']
    return f"{format_value(summary['mean'], kind)} ± {format_value(summary['std'], kind)}"


def render_table(title, columns, rows):
    html = [f'<h3>{title}</h3>']
    html.append('<table border="1" cellpadding="6" cellspacing="0">')
    html.append('<thead><tr>' + ''.join(f'<th>{col}</th>' for col in columns) + '</tr></thead>')
    html.append('<tbody>')
    for row in rows:
        html.append('<tr>' + ''.join(f'<td>{cell}</td>' for cell in row) + '</tr>')
    html.append('</tbody></table>')
    display(HTML(''.join(html)))


## Headline endpoint result: final Hessian trace

The original anti-sharpness claim in this pair should be judged against the **actual endpoint trace comparison**. The primary decision output uses the script's current statistic: the **ratio of mean traces** `mean(trace_muon) / mean(trace_sgd)`.


In [ ]:
trace_rows = []
for arch_key in ARCH_KEYS:
    summary = metric_summary(arch_key, 'trace')
    test = results['architectures'][arch_key]['decision_outputs']['trace_hypothesis']
    trace_rows.append([
        arch_key,
        mean_std_text(summary['sgd'], 'trace'),
        mean_std_text(summary['muon'], 'trace'),
        format_value(summary['ratio_of_means'], 'float'),
        format_value(summary['mean_seedwise_ratio'], 'float'),
        f"{test['muon_lower_trace_seeds']}/{test['n_seeds']}",
        test['verdict'],
    ])

render_table(
    title='Final Hessian trace summary',
    columns=['Architecture', 'SGD mean ± std', 'Muon mean ± std', 'Ratio of means', 'Mean seedwise ratio', 'Muon < SGD seeds', 'Verdict'],
    rows=trace_rows,
)

fig, axes = plt.subplots(1, len(ARCH_KEYS), figsize=(6 * len(ARCH_KEYS), 4.8), constrained_layout=True)
if len(ARCH_KEYS) == 1:
    axes = [axes]

for ax, arch_key in zip(axes, ARCH_KEYS):
    sgd_trace = per_seed_metric(arch_key, 'sgd', 'trace')
    muon_trace = per_seed_metric(arch_key, 'muon', 'trace')
    lo = min(np.min(sgd_trace), np.min(muon_trace))
    hi = max(np.max(sgd_trace), np.max(muon_trace))
    ax.scatter(sgd_trace, muon_trace, s=55, alpha=0.8)
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1.5, label='y = x')
    ax.set_title(f'{arch_key}: per-seed trace')
    ax.set_xlabel('SGD final trace')
    ax.set_ylabel('Muon final trace')
    ax.legend()
    ax.grid(alpha=0.25)

plt.show()


In [ ]:
trace_lines = ['### Trace interpretation', '']
for arch_key in ARCH_KEYS:
    summary = metric_summary(arch_key, 'trace')
    test = results['architectures'][arch_key]['decision_outputs']['trace_hypothesis']
    trace_lines.append(
        f"- **{arch_key}**: ratio of mean traces = `{summary['ratio_of_means']:.4f}`, "
        f"mean seedwise trace ratio = `{summary['mean_seedwise_ratio']:.4f}`, "
        f"Muon has lower trace on `{test['muon_lower_trace_seeds']}/{test['n_seeds']}` seeds, "
        f"so the script's main trace hypothesis is **{test['verdict']}**."
    )
trace_lines.extend([
    '',
    'This notebook therefore reports the toy anti-sharpness test as a **question under evaluation**, not as an already-established flatter-minima result.'
])
display(Markdown('\n'.join(trace_lines)))


## Convergence speed and final loss quality

The stronger curvature claim should be separated from the simpler optimization result: **how quickly each optimizer reaches the stopping threshold**.


In [ ]:
speed_rows = []
for arch_key in ARCH_KEYS:
    step_summary = metric_summary(arch_key, 'steps')
    loss_summary = metric_summary(arch_key, 'loss')
    speed_test = results['architectures'][arch_key]['decision_outputs']['speed_comparison']
    speed_rows.append([
        arch_key,
        mean_std_text(step_summary['sgd'], 'steps'),
        mean_std_text(step_summary['muon'], 'steps'),
        format_value(step_summary['ratio_of_means'], 'float'),
        f"{speed_test['muon_fewer_steps_seeds']}/{speed_test['n_seeds']}",
        mean_std_text(loss_summary['sgd'], 'loss'),
        mean_std_text(loss_summary['muon'], 'loss'),
    ])

render_table(
    title='Convergence summary',
    columns=['Architecture', 'SGD steps mean ± std', 'Muon steps mean ± std', 'Mean-steps ratio', 'Muon fewer-step seeds', 'SGD final loss mean ± std', 'Muon final loss mean ± std'],
    rows=speed_rows,
)

fig, axes = plt.subplots(1, len(ARCH_KEYS), figsize=(6 * len(ARCH_KEYS), 4.8), constrained_layout=True)
if len(ARCH_KEYS) == 1:
    axes = [axes]

for ax, arch_key in zip(axes, ARCH_KEYS):
    sgd_steps = per_seed_metric(arch_key, 'sgd', 'steps')
    muon_steps = per_seed_metric(arch_key, 'muon', 'steps')
    for s, m in zip(sgd_steps, muon_steps):
        ax.plot(['SGD', 'Muon'], [s, m], marker='o', alpha=0.45)
    ax.set_title(f'{arch_key}: steps to threshold')
    ax.set_ylabel('Training steps')
    ax.grid(alpha=0.25)

plt.show()


In [ ]:
speed_lines = ['### Speed interpretation', '']
for arch_key in ARCH_KEYS:
    speed_test = results['architectures'][arch_key]['decision_outputs']['speed_comparison']
    speed_lines.append(
        f"- **{arch_key}**: mean-steps ratio = `{speed_test['ratio_of_means']:.4f}`, "
        f"mean seedwise step ratio = `{speed_test['mean_seedwise_ratio']:.4f}`, "
        f"Muon uses fewer steps on `{speed_test['muon_fewer_steps_seeds']}/{speed_test['n_seeds']}` seeds, "
        f"so the speed verdict is **{speed_test['verdict']}**."
    )
speed_lines.extend([
    '',
    'This is a distinct claim from flatter minima: faster convergence does **not** by itself imply a lower final Hessian trace.'
])
display(Markdown('\n'.join(speed_lines)))


## Additional endpoint curvature summaries

To make the pair more academically self-explanatory, the notebook also presents endpoint summaries for:

- **$\lambda_{\max}$**: largest Hessian eigenvalue
- **$\kappa$**: condition number based on the smallest positive eigenvalue above tolerance
- **near-zero eigenvalue count proxy**: threshold-based proxy only
- **negative-curvature diagnostics**: count and total mass of negative eigenvalues


In [ ]:
extra_metrics = ['lambda_max', 'kappa', 'near_zero_count_proxy', 'n_negative', 'negative_mass_ratio']
extra_rows = []
for arch_key in ARCH_KEYS:
    for metric in extra_metrics:
        summary = metric_summary(arch_key, metric)
        extra_rows.append([
            arch_key,
            anti.METRIC_DISPLAY[metric]['label'],
            mean_std_text(summary['sgd'], metric),
            mean_std_text(summary['muon'], metric),
            format_value(summary['ratio_of_means'], 'float'),
        ])

render_table(
    title='Additional curvature and diagnostic summaries',
    columns=['Architecture', 'Metric', 'SGD mean ± std', 'Muon mean ± std', 'Ratio of means'],
    rows=extra_rows,
)

ratio_metrics = ['lambda_max', 'kappa', 'near_zero_count_proxy']
x = np.arange(len(ratio_metrics))
width = 0.36

fig, ax = plt.subplots(figsize=(8.2, 4.8), constrained_layout=True)
for i, arch_key in enumerate(ARCH_KEYS):
    offsets = x + (i - (len(ARCH_KEYS) - 1) / 2) * width
    ratios = [metric_summary(arch_key, metric)['ratio_of_means'] for metric in ratio_metrics]
    ax.bar(offsets, ratios, width=width, label=arch_key)

ax.axhline(1.0, color='k', linestyle='--', linewidth=1.2, label='parity')
ax.set_xticks(x)
ax.set_xticklabels([anti.METRIC_DISPLAY[m]['label'] for m in ratio_metrics], rotation=10)
ax.set_ylabel('Muon / SGD ratio of means')
ax.set_title('Endpoint curvature summaries beyond trace')
ax.legend()
ax.grid(axis='y', alpha=0.25)
plt.show()


In [ ]:
hessian_rows = []
for arch_key in ARCH_KEYS:
    sgd_asym = np.asarray([
        seed_result['sgd']['hessian_fd']['pre_symmetry_max_abs_asymmetry']
        for seed_result in seed_results(arch_key)
    ], dtype=float)
    muon_asym = np.asarray([
        seed_result['muon']['hessian_fd']['pre_symmetry_max_abs_asymmetry']
        for seed_result in seed_results(arch_key)
    ], dtype=float)
    hessian_rows.append([
        arch_key,
        f"{np.mean(sgd_asym):.4e} ± {np.std(sgd_asym):.2e}",
        f"{np.mean(muon_asym):.4e} ± {np.std(muon_asym):.2e}",
        f"eps = {CONFIG['hessian_eps']}",
    ])

render_table(
    title='Finite-difference Hessian quality check',
    columns=['Architecture', 'SGD pre-symmetry max |H-H^T|', 'Muon pre-symmetry max |H-H^T|', 'FD step'],
    rows=hessian_rows,
)

display(Markdown(
    """
**Caveat.** The notebook reports a **full finite-difference Hessian estimate**, not an analytic Hessian. The pre-symmetry asymmetry diagnostic is included so the reader can see the size of the raw finite-difference non-symmetry before explicit symmetrization.

A second caveat is that **$\kappa$ can be fragile** here, because it depends on the smallest positive eigenvalue that clears a tolerance threshold.
"""
))


## Calibrated conclusion

The closing summary should distinguish clearly between:

1. **optimization speed**
2. **endpoint curvature**
3. stronger mechanistic or gauge-theoretic claims that this toy study does not directly establish


In [ ]:
conclusion_lines = ['### Final conclusion', '']
for arch_key in ARCH_KEYS:
    trace_test = results['architectures'][arch_key]['decision_outputs']['trace_hypothesis']
    speed_test = results['architectures'][arch_key]['decision_outputs']['speed_comparison']
    near_zero_ratio = metric_summary(arch_key, 'near_zero_count_proxy')['ratio_of_means']
    lambda_ratio = metric_summary(arch_key, 'lambda_max')['ratio_of_means']
    conclusion_lines.append(
        f"- **{arch_key}**: Muon is `{speed_test['verdict']}` by the current steps-to-threshold metric "
        f"(mean-steps ratio `{speed_test['ratio_of_means']:.4f}`), while the endpoint trace hypothesis is "
        f"**{trace_test['verdict']}** (trace ratio `{trace_test['ratio_of_means']:.4f}`). "
        f"For context, `lambda_max` ratio is `{lambda_ratio:.4f}` and the near-zero-count proxy ratio is `{near_zero_ratio:.4f}`."
    )

conclusion_lines.extend([
    '',
    f"- **Across architectures:** {results['overall']['summary_statement']}",
    f"- **Depth comparison:** {results['overall']['depth_comparison']['interpretation']}",
    '- **Interpretation boundary:** this notebook supports claims about endpoint finite-difference Hessian summaries in a toy deep-linear setting. It does **not** by itself establish trajectory anti-sharpness, true gauge-mode counting, or generalization advantages.',
])
display(Markdown('\n'.join(conclusion_lines)))
